# 02 — EDGAR Generalization Test: Johnson & Johnson (JNJ)

**Phase 0 goal:** confirm the Apple parsing pipeline generalizes to a different sector and fiscal calendar.

JNJ is a healthcare mega-cap with a calendar fiscal year (ends Dec 31), vs. Apple's September year-end.
This notebook runs the same extraction pipeline against CIK `0000200406` and reports any tagging differences.

## 0. Setup

In [ ]:
import hashlib
import json
import pathlib
import time
import datetime
import os

import httpx
import pandas as pd
from dotenv import load_dotenv

load_dotenv(dotenv_path=pathlib.Path(".env"))

HEADERS = {"User-Agent": os.environ["EDGAR_USER_AGENT"]}
CACHE_DIR = pathlib.Path(".edgar_cache")
CACHE_DIR.mkdir(exist_ok=True)

_last_request: float = 0.0

def fetch_json(url: str) -> dict:
    """Fetch URL with disk cache and ≤10 req/sec rate limit."""
    global _last_request
    key = hashlib.sha256(url.encode()).hexdigest()[:16]
    cache_file = CACHE_DIR / f"{key}.json"

    if cache_file.exists():
        return json.loads(cache_file.read_text())

    elapsed = time.monotonic() - _last_request
    if elapsed < 0.1:
        time.sleep(0.1 - elapsed)

    resp = httpx.get(url, headers=HEADERS, timeout=30)
    resp.raise_for_status()
    _last_request = time.monotonic()

    data = resp.json()
    cache_file.write_text(json.dumps(data))
    return data

print("Setup complete. User-Agent:", HEADERS["User-Agent"])

## 0.5. Smoke Test

Verifies three things before any real work:
1. `company_tickers.json` returns valid JSON
2. The response was written to `.edgar_cache/`
3. A second call hits the cache (no network)

In [ ]:
TICKERS_URL = "https://www.sec.gov/files/company_tickers.json"

t0 = time.monotonic()
tickers = fetch_json(TICKERS_URL)
first_call_ms = (time.monotonic() - t0) * 1000

assert isinstance(tickers, dict), "Expected a dict"
assert len(tickers) > 0, "Tickers dict is empty"

key = hashlib.sha256(TICKERS_URL.encode()).hexdigest()[:16]
cache_file = CACHE_DIR / f"{key}.json"
assert cache_file.exists(), f"Cache file not found: {cache_file}"

t1 = time.monotonic()
tickers2 = fetch_json(TICKERS_URL)
second_call_ms = (time.monotonic() - t1) * 1000

assert tickers == tickers2

print(f"PASS — {len(tickers):,} tickers loaded")
print(f"First call:  {first_call_ms:.1f} ms")
print(f"Second call: {second_call_ms:.1f} ms")
print(f"Cache file:  {cache_file}")


## 1. Locate JNJ's CIK

In [ ]:
jnj = next(v for v in tickers.values() if v["ticker"] == "JNJ")
CIK = str(jnj["cik_str"]).zfill(10)

print(f"Ticker : {jnj['ticker']}")
print(f"Name   : {jnj['title']}")
print(f"CIK    : {CIK}")

## 2. Find the Most Recent 10-Q

In [ ]:
SUBMISSIONS_URL = f"https://data.sec.gov/submissions/CIK{CIK}.json"
submissions = fetch_json(SUBMISSIONS_URL)

recent = submissions["filings"]["recent"]
tenq_filings = [
    (date, accn)
    for form, date, accn in zip(
        recent["form"], recent["filingDate"], recent["accessionNumber"]
    )
    if form == "10-Q"
]

most_recent_date, most_recent_accn = tenq_filings[0]

print(f"Most recent 10-Q filed : {most_recent_date}")
print(f"Accession number       : {most_recent_accn}")

## 3. Pull All Company Facts

In [ ]:
FACTS_URL = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{CIK}.json"
print("Fetching company facts (may take a moment)...")
facts = fetch_json(FACTS_URL)

us_gaap = facts["facts"]["us-gaap"]

print(f"Top-level keys       : {list(facts.keys())}")
print(f"Taxonomies available : {list(facts['facts'].keys())}")
print(f"us-gaap concepts     : {len(us_gaap):,}")

## 4. Extract Revenue (Last 4 Quarters)

In [ ]:
import datetime

def parse_date(s: str) -> datetime.date:
    return datetime.date.fromisoformat(s)

def _dedup_by_end(facts_list: list, lo: int, hi: int) -> list:
    """Return one fact per end-date within a period-length window.
    Takes the latest-filed entry per end date. Uses end date (not fy) as
    period identifier — fy labels in EDGAR can be wrong for multi-year
    comparative statements in 10-Ks."""
    by_end: dict = {}
    for f in facts_list:
        start = f.get("start")
        end   = f.get("end")
        if not start or not end:
            continue
        if f.get("form") not in ("10-Q", "10-K"):
            continue
        days = (parse_date(end) - parse_date(start)).days
        if not (lo <= days <= hi):
            continue
        if end not in by_end or f["filed"] > by_end[end]["filed"]:
            by_end[end] = f
    return sorted(by_end.values(), key=lambda x: x["end"])

def filter_quarterly(facts_list: list) -> list:
    """Standalone quarterly Income Statement facts (70–100 day periods).
    Deduplicates by end date to handle EDGAR multi-year comparison labels."""
    return _dedup_by_end(facts_list, 70, 100)

def derive_q4(facts_list: list) -> list:
    """Derive Q4 = FY annual − 9M YTD for Income Statement items.
    Groups by fiscal-year start date (shared by FY annual and Q3 YTD),
    which avoids fy-label collisions in EDGAR comparative statements."""
    annual = _dedup_by_end(facts_list, 340, 380)
    ytd_9m = _dedup_by_end(facts_list, 250, 290)

    ann_by_start = {f["start"]: f for f in annual}
    ytd_by_start = {f["start"]: f for f in ytd_9m}

    q4s = []
    for start, ann in ann_by_start.items():
        if start in ytd_by_start:
            ytd = ytd_by_start[start]
            q4s.append({
                "fy": ann.get("fy"), "fp": "Q4",
                "val": ann["val"] - ytd["val"],
                "end": ann["end"], "filed": ann["filed"],
            })
    return q4s

def derive_cashflow_quarterly(facts_list: list) -> list:
    """Un-cumulate YTD cash flow items into standalone quarters.
    Groups all period types by their shared fiscal-year start date:
    Q1 (~90d), H1 (~181d), 9M (~272d), FY (~363d) all share the same start.
    Derives: Q2 = H1−Q1, Q3 = 9M−H1, Q4 = FY−9M."""
    q1s  = _dedup_by_end(facts_list,  75, 105)
    h1s  = _dedup_by_end(facts_list, 165, 200)
    m9s  = _dedup_by_end(facts_list, 255, 290)
    anns = _dedup_by_end(facts_list, 340, 380)

    q1_by_start  = {f["start"]: f for f in q1s}
    h1_by_start  = {f["start"]: f for f in h1s}
    m9_by_start  = {f["start"]: f for f in m9s}
    ann_by_start = {f["start"]: f for f in anns}

    all_starts = set(q1_by_start) | set(h1_by_start) | set(m9_by_start) | set(ann_by_start)
    results = []
    for start in all_starts:
        q1  = q1_by_start.get(start)
        h1  = h1_by_start.get(start)
        m9  = m9_by_start.get(start)
        ann = ann_by_start.get(start)
        fy  = (ann or m9 or h1 or q1).get("fy")  # prefer native FY labels over later comparative Q1 facts
        if q1:
            results.append({"fy": fy, "fp": "Q1", "val": q1["val"],              "end": q1["end"],  "filed": q1["filed"]})
        if h1 and q1:
            results.append({"fy": fy, "fp": "Q2", "val": h1["val"] - q1["val"],  "end": h1["end"],  "filed": h1["filed"]})
        if m9 and h1:
            results.append({"fy": fy, "fp": "Q3", "val": m9["val"] - h1["val"],  "end": m9["end"],  "filed": m9["filed"]})
        if ann and m9:
            results.append({"fy": fy, "fp": "Q4", "val": ann["val"] - m9["val"], "end": ann["end"], "filed": ann["filed"]})
    return sorted(results, key=lambda x: x["end"])

# --- Revenue ---
rev_facts = us_gaap["RevenueFromContractWithCustomerExcludingAssessedTax"]["units"]["USD"]

rev_quarterly = filter_quarterly(rev_facts)
rev_q4 = derive_q4(rev_facts)
all_rev = sorted(rev_quarterly + rev_q4, key=lambda x: x["end"])[-4:]

df_rev = pd.DataFrame([
    {"Period": f"FY{r['fy']} {r['fp']}", "Revenue ($M)": round(r["val"] / 1e6, 1)}
    for r in all_rev
])
df_rev

## 5. Extract Net Income (Last 4 Quarters)

In [ ]:
ni_facts = us_gaap["NetIncomeLoss"]["units"]["USD"]

all_ni = sorted(filter_quarterly(ni_facts) + derive_q4(ni_facts), key=lambda x: x["end"])[-4:]

df_ni = pd.DataFrame([
    {"Period": f"FY{r['fy']} {r['fp']}", "Net Income ($M)": round(r["val"] / 1e6, 1)}
    for r in all_ni
])
df_ni

## 6. Extract EPS (Last 4 Quarters)

In [ ]:
eps_basic_facts   = us_gaap["EarningsPerShareBasic"]["units"]["USD/shares"]
eps_diluted_facts = us_gaap["EarningsPerShareDiluted"]["units"]["USD/shares"]

all_basic   = sorted(filter_quarterly(eps_basic_facts)   + derive_q4(eps_basic_facts),   key=lambda x: x["end"])[-4:]
all_diluted = sorted(filter_quarterly(eps_diluted_facts) + derive_q4(eps_diluted_facts), key=lambda x: x["end"])[-4:]

basic_map   = {f"FY{r['fy']} {r['fp']}": round(r["val"], 4) for r in all_basic}
diluted_map = {f"FY{r['fy']} {r['fp']}": round(r["val"], 4) for r in all_diluted}
periods = sorted(set(basic_map) | set(diluted_map))

df_eps = pd.DataFrame([
    {
        "Period":          p,
        "EPS Basic ($)":   basic_map.get(p),
        "EPS Diluted ($)": diluted_map.get(p),
    }
    for p in periods
])
df_eps

## 7. Derive Free Cash Flow (Last 4 Quarters)

In [ ]:
opcf_facts  = us_gaap["NetCashProvidedByUsedInOperatingActivities"]["units"]["USD"]
capex_facts = us_gaap["PaymentsToAcquirePropertyPlantAndEquipment"]["units"]["USD"]

# Cash flow items may be cumulative YTD — use derive_cashflow_quarterly
all_opcf  = sorted(derive_cashflow_quarterly(opcf_facts),  key=lambda x: x["end"])
all_capex = sorted(derive_cashflow_quarterly(capex_facts), key=lambda x: x["end"])

opcf_map  = {f"FY{r['fy']} {r['fp']}": r["val"] for r in all_opcf}
capex_map = {f"FY{r['fy']} {r['fp']}": r["val"] for r in all_capex}
periods = sorted(set(opcf_map) | set(capex_map))

df_fcf = pd.DataFrame([
    {
        "Period":    p,
        "Op CF ($M)": round(opcf_map[p]  / 1e6, 1) if p in opcf_map  else None,
        "CapEx ($M)": round(capex_map[p] / 1e6, 1) if p in capex_map else None,
        "FCF ($M)":   round((opcf_map[p] - capex_map[p]) / 1e6, 1)
                      if p in opcf_map and p in capex_map else None,
    }
    for p in periods
])
df_fcf

## 8. Combined Summary Table

In [ ]:
canonical   = [f"FY{r['fy']} {r['fp']}" for r in all_rev]

ni_map      = {f"FY{r['fy']} {r['fp']}": round(r["val"] / 1e6, 1) for r in all_ni}
basic_map   = {f"FY{r['fy']} {r['fp']}": round(r["val"], 4)        for r in all_basic}
diluted_map = {f"FY{r['fy']} {r['fp']}": round(r["val"], 4)        for r in all_diluted}
fcf_map     = {row["Period"]: row["FCF ($M)"] for row in df_fcf.to_dict("records")}
rev_map     = {f"FY{r['fy']} {r['fp']}": round(r["val"] / 1e6, 1) for r in all_rev}

df_summary = pd.DataFrame([
    {
        "Period":           p,
        "Revenue ($M)":     rev_map.get(p),
        "Net Income ($M)":  ni_map.get(p),
        "EPS Basic ($)":    basic_map.get(p),
        "EPS Diluted ($)":  diluted_map.get(p),
        "FCF ($M)":         fcf_map.get(p),
    }
    for p in canonical
])
df_summary

In [ ]:
output_path = pathlib.Path("jnj_phase0_output.csv")
df_summary.to_csv(output_path, index=False)
print(f"Saved {output_path} ({len(df_summary)} rows)")

## 9. Phase 0 Generalization Observations

### Did the same XBRL concepts work?

Yes. JNJ used the same concepts as Apple for the core Phase 0 metrics:

- Revenue: `RevenueFromContractWithCustomerExcludingAssessedTax`
- Net income: `NetIncomeLoss`
- EPS: `EarningsPerShareBasic` / `EarningsPerShareDiluted` under `USD/shares`
- Operating cash flow: `NetCashProvidedByUsedInOperatingActivities`
- CapEx: `PaymentsToAcquirePropertyPlantAndEquipment`

### Did the cumulative-YTD cash flow pattern hold or differ?

It held. JNJ cash flow facts are YTD for Q2 and Q3, with FY annual in the 10-K. The same `derive_cashflow_quarterly` approach works: Q2 = H1 - Q1, Q3 = 9M - H1, Q4 = FY - 9M.

### Any new tagging variations vs. Apple?

No new concept tags were required, but JNJ exposed a label-quality issue in duplicate facts. The 2026 Q1 filing repeats the 2025 Q1 comparative cash-flow period with `fy=2026`, so cash-flow derivation must prefer the annual/9M/H1 fiscal-year labels for a shared fiscal-year start, not the latest-filed duplicate Q1 label.

### Net verdict: is the pipeline generalizing or Apple-specific?

- [x] **Generalizing** — same helpers work, minor tag/label handling only
- [ ] **Apple-specific** — significant rework needed for other filers

_Reasoning:_ JNJ is a different sector with a calendar fiscal year, and the same concept set plus period-length logic produced the expected recent-quarter summary after fixing the duplicate comparative Q1 FY label issue. This supports continuing self-built EDGAR into Phase 1, with the concept-mapping layer made explicit before scaling beyond hand notebooks.
